# ⚽ FM Save Copilot — v2

Welcome! This notebook turns your Football Manager 2024 squad into a 10-section Director of Football report — no installation needed.

**What you'll need:**
1. Your squad exported from FM24 as an HTML file ([export guide](https://github.com/laweh-dev/fm24-sporting-director/blob/main/VIEW-SETUP.md))
2. *(Optional)* An Anthropic API key for the written narrative — costs **~$0.05 per report** using Claude Sonnet

**How to use:** Run each cell in order (click ▶️ or use **Runtime → Run all**).

---

**Report sections:** Executive Summary · Current Squad · Squad Depth · Priority Areas · Priority Signings · Young Talent · Decline & Contract Risks · Financial Audit · Who Must Be Sold · Strategic Outlook

In [ ]:
#@title Step 1: Install the tool (click ▶️, wait ~60 seconds) { display-mode: "form" }
import os, sys

# Clone on first run; pull latest on subsequent runs
if not os.path.exists('fm24-sporting-director'):
    !git clone -q https://github.com/laweh-dev/fm24-sporting-director.git

%cd fm24-sporting-director
!git pull -q  # always fetch the latest version
!pip install -q -e . 2>&1 | tail -3

# Make fm_copilot importable directly (survives if Colab restarts after install)
_src = os.path.join(os.path.abspath('.'), 'src')
if _src not in sys.path:
    sys.path.insert(0, _src)

print('✅ Installed and up to date! Move to the next step.')


## Step 2: Upload your FM24 exports

Run the next cell and click **Choose Files** to upload:
- Your **squad export** — save it as `squad.html` from the FM24 Squad screen
- Your **market export** — *(optional)* save as `market.html` from the Scouting screen

Don't have these yet? Follow the [export guide](https://github.com/laweh-dev/fm24-sporting-director/blob/main/VIEW-SETUP.md) first.

In [ ]:
#@title Step 2: Upload your squad files { display-mode: "form" }
from google.colab import files
import os, shutil

os.makedirs('data_uploads', exist_ok=True)
print('Upload your squad.html (and optionally market.html):')
uploaded = files.upload()

for filename in uploaded:
    dest = f'data_uploads/{filename}'
    shutil.move(filename, dest)
    print(f'  ✅ Saved to {dest}')

if 'squad.html' not in [os.path.basename(f) for f in os.listdir('data_uploads')]:
    print('⚠️  No squad.html found — make sure you name the file squad.html')
else:
    print('\nReady for Step 3!')

## Step 3: Tell the DoF about your club

Fill in the dropdowns and checkboxes — the DoF will auto-compose your tactical profile and squad assessment from your selections.

> **Priority positions:** tick the positions you think need strengthening. The DoF will compare this against the data and push back where it disagrees.

In [ ]:
#@title Step 3: Configure your club { display-mode: "form" }

# ── Club details ─────────────────────────────────────────────────────────
club_name    = 'My Club'           #@param {type:"string"}
league       = 'Championship'      #@param {type:"string"}
competitions = 'FA Cup, Carabao Cup'  #@param {type:"string"}
fm_season    = '2025/26'           #@param {type:"string"}
transfer_budget = '15m'            #@param {type:"string"}
wage_budget     = '100k/week'      #@param {type:"string"}
board_objective = "Comfortable mid-table"  #@param ["Win the league", "Challenge for promotion / top 6", "Comfortable mid-table", "Avoid relegation", "Survive — any points will do", "Win a cup", "Qualify for Europe"]
dof_mode        = "edwards"        #@param ["edwards", "monchi", "edu"]

# ── Formation ─────────────────────────────────────────────────────────────
formation = "4-2-3-1"  #@param ["4-2-3-1", "4-3-3", "4-4-2", "3-5-2", "4-1-4-1", "5-3-2", "3-4-3", "4-2-2-2"]

# ── Role choices (pick the role played in each slot of your formation) ────
# GK slot
gk_role   = "Sweeper Keeper"        #@param ["Sweeper Keeper", "Goalkeeper"]
# Full-back / wide-defender slot (RB / LB)
fb_role   = "Inverted Full Back"    #@param ["Inverted Full Back", "Full Back", "Wing Back", "Complete Wing Back"]
# Wing-back slot (RWB / LWB in 3/5 back formations)
wb_role   = "Complete Wing Back"    #@param ["Wing Back", "Complete Wing Back", "Inverted Wing Back"]
# Centre-back slot
cb_role   = "Central Defender"      #@param ["Central Defender", "Ball-Playing Defender", "No-Nonsense CB", "Wide Centre Back"]
# Defensive midfielder slot
dm_role   = "Half Back"             #@param ["Half Back", "Anchor", "Defensive Midfielder", "Ball-Winning Midfielder", "Deep-Lying Playmaker", "Regista"]
# Central midfielder slot
cm_role   = "Box-to-Box Midfielder" #@param ["Box-to-Box Midfielder", "Mezzala", "Carrilero", "Central Midfielder", "Advanced Playmaker", "Roaming Playmaker", "Ball-Winning Midfielder", "Segundo Volante"]
# Attacking midfielder slot (AM / 10)
am_role   = "Advanced Playmaker"    #@param ["Advanced Playmaker", "Shadow Striker", "Enganche", "Trequartista", "Attacking Midfielder"]
# Wide forward / winger slot
wide_role = "Inverted Winger"       #@param ["Inverted Winger", "Winger", "Inside Forward", "Defensive Winger", "Wide Playmaker", "Raumdeuter"]
# Striker slot
st_role   = "Advanced Forward"      #@param ["Advanced Forward", "Pressing Forward", "Complete Forward", "Deep-Lying Forward", "False Nine", "Target Forward", "Poacher"]

# ── Tactical style ────────────────────────────────────────────────────────
pressing  = "High press — gegenpress, win ball high up the pitch"  #@param ["High press — gegenpress, win ball high up the pitch", "High press — compact mid-block, press on trigger", "Moderate press — sit in shape, press only when ahead", "Low block — defend deep, absorb and hit on counter", "Low block — park the bus, rarely press above halfway"]
build_up  = "Short passing — patient possession through the thirds"  #@param ["Short passing — patient possession through the thirds", "Mixed — short when possible, direct when needed", "Direct — quick vertical passing, exploit space in behind", "Long ball — bypass midfield, use physical target striker", "Counter-attack — sit off, win ball, transition fast"]
def_line  = "Standard — push up to press, drop to hold shape"  #@param ["High line — offside trap, squeeze the pitch", "Standard — push up to press, drop to hold shape", "Deep — hold a low block, limit space in behind"]
width     = "Standard width — balance between central play and wide options"  #@param ["Narrow — central overloads, inverted wingers cut inside", "Standard width — balance between central play and wide options", "Wide — attack the flanks, lots of crossing"]

# ── Priority positions (tick the ones you think need strengthening) ───────
need_gk  = False  #@param {type:"boolean"}
need_rb  = False  #@param {type:"boolean"}
need_cb  = False  #@param {type:"boolean"}
need_lb  = False  #@param {type:"boolean"}
need_dm  = False  #@param {type:"boolean"}
need_cm  = False  #@param {type:"boolean"}
need_am  = False  #@param {type:"boolean"}
need_rw  = False  #@param {type:"boolean"}
need_lw  = False  #@param {type:"boolean"}
need_st  = False  #@param {type:"boolean"}

# ── Overall squad concerns ─────────────────────────────────────────────────
squad_depth    = "Thin in one or two positions"  #@param ["Depth is fine overall", "Thin in one or two positions", "Major gaps — multiple positions short", "Too many players not yet ready", "Too top-heavy, limited rotation"]
squad_age      = "Good mix of youth and experience"  #@param ["Too many aging players — rebuild needed", "Good mix of youth and experience", "Too young — lack experienced heads", "Prime-age heavy — good now, need to plan ahead", "Balanced — no major age concerns"]

# ── Auto-compose tactical_direction and user_squad_read ───────────────────
tactical_direction = (
    f"{formation} system. "
    f"Roles: GK={gk_role}, FB={fb_role}, CB={cb_role}, DM={dm_role}, "
    f"CM={cm_role}, Wide={wide_role}, ST={st_role}. "
    f"Pressing: {pressing.split(' — ')[0].lower()}. "
    f"Build-up: {build_up.split(' — ')[0].lower()}. "
    f"Defensive line: {def_line.split(' — ')[0].lower()}. "
    f"Width: {width.split(' — ')[0].lower()}."
)

_pos_map = {'GK': need_gk, 'RB': need_rb, 'CB': need_cb, 'LB': need_lb,
            'DM': need_dm, 'CM': need_cm, 'AM': need_am, 'RW': need_rw,
            'LW': need_lw, 'ST': need_st}
_priority_pos = [pos for pos, needed in _pos_map.items() if needed]
_pos_str = ', '.join(_priority_pos) if _priority_pos else 'no specific positions flagged'
user_squad_read = (
    f"Priority signings: {_pos_str}. "
    f"Depth concern: {squad_depth.lower()}. "
    f"Age profile: {squad_age.lower()}."
)

# Map display names → role_key strings expected by analysis.py
_ROLE_KEYS = {
    "Sweeper Keeper": "sweeper_keeper", "Goalkeeper": "goalkeeper",
    "Inverted Full Back": "inverted_full_back", "Full Back": "full_back",
    "Wing Back": "wing_back", "Complete Wing Back": "complete_wing_back",
    "Inverted Wing Back": "inverted_wing_back",
    "Central Defender": "central_defender", "Ball-Playing Defender": "ball_playing_defender",
    "No-Nonsense CB": "no_nonsense_centre_back", "Wide Centre Back": "wide_centre_back",
    "Half Back": "half_back", "Anchor": "anchor",
    "Defensive Midfielder": "defensive_midfielder",
    "Ball-Winning Midfielder": "ball_winning_midfielder",
    "Deep-Lying Playmaker": "deep_lying_playmaker", "Regista": "regista",
    "Box-to-Box Midfielder": "box_to_box_midfielder", "Mezzala": "mezzala",
    "Carrilero": "carrilero", "Central Midfielder": "central_midfielder",
    "Advanced Playmaker": "advanced_playmaker", "Roaming Playmaker": "roaming_playmaker",
    "Segundo Volante": "segundo_volante",
    "Shadow Striker": "shadow_striker", "Enganche": "enganche",
    "Trequartista": "trequartista", "Attacking Midfielder": "attacking_midfielder",
    "Inverted Winger": "inverted_winger", "Winger": "winger",
    "Inside Forward": "inside_forward", "Defensive Winger": "defensive_winger",
    "Wide Playmaker": "wide_playmaker", "Raumdeuter": "raumdeuter",
    "Advanced Forward": "advanced_forward", "Pressing Forward": "pressing_forward",
    "Complete Forward": "complete_forward", "Deep-Lying Forward": "deep_lying_forward",
    "False Nine": "false_nine", "Target Forward": "target_forward", "Poacher": "poacher",
}

# ─────────────────────────────────────────────────────────────────────────
import sys, os as _os
_src = _os.path.join(_os.path.abspath('.'), 'src')
if _src not in sys.path:
    sys.path.insert(0, _src)

from fm_copilot.wizard import generate_context_from_form
generate_context_from_form({
    'club_name':          club_name,
    'league':             league,
    'competitions':       competitions,
    'fm_season':          fm_season,
    'transfer_budget':    transfer_budget,
    'wage_budget':        wage_budget,
    'board_objective':    board_objective,
    'tactical_direction': tactical_direction,
    'user_squad_read':    user_squad_read,
    'dof_mode':           dof_mode,
    'formation':          formation,
    'gk_role':            _ROLE_KEYS.get(gk_role,   'sweeper_keeper'),
    'fb_role':            _ROLE_KEYS.get(fb_role,   'inverted_full_back'),
    'wb_role':            _ROLE_KEYS.get(wb_role,   'complete_wing_back'),
    'cb_role':            _ROLE_KEYS.get(cb_role,   'central_defender'),
    'dm_role':            _ROLE_KEYS.get(dm_role,   'half_back'),
    'cm_role':            _ROLE_KEYS.get(cm_role,   'box_to_box_midfielder'),
    'am_role':            _ROLE_KEYS.get(am_role,   'advanced_playmaker'),
    'wide_role':          _ROLE_KEYS.get(wide_role, 'inverted_winger'),
    'st_role':            _ROLE_KEYS.get(st_role,   'advanced_forward'),
}, context_dir='context')

print(f'\n✅ {club_name} ({league}) — {dof_mode.title()} mode')
print(f'   Formation: {formation}  |  GK: {gk_role}  |  FB: {fb_role}  |  CB: {cb_role}')
print(f'   DM: {dm_role}  |  CM: {cm_role}  |  Wide: {wide_role}  |  ST: {st_role}')
print(f'   Squad read: {user_squad_read}')


## Step 4: Add your API key *(optional — skip for free mode)*

For the written Director of Football narrative, paste your Anthropic API key below.
Get one at [console.anthropic.com](https://console.anthropic.com) — costs **~$0.05 per report** using Claude Sonnet 4.6.

**Skip this** to get free mode: all the scoring, depth tables, radar charts, and sell/contract data — just without the written narrative sections.

> Your key is entered securely via `getpass` and is not stored in the notebook.

In [ ]:
#@title Step 4: Enter your API key (or press Enter to skip) { display-mode: "form" }
import getpass
api_key = getpass.getpass('Paste your Anthropic API key (or press Enter to skip): ')
if api_key:
    print('✅ Key received (not shown). Will generate full written report.')
else:
    print('Running in free mode — analysis and charts, no written narrative.')

## Step 5: Generate your report! 🎉

Run the next cell. It'll:
1. Parse your squad file and run the analysis (free, on Google's servers)
2. If you added an API key: call Claude Sonnet to write the 10-section narrative (~$0.05)

Takes 15–90 seconds depending on squad size and whether AI is enabled.

In [ ]:
#@title Step 5: Generate the report { display-mode: "form" }
import sys, os
_src = os.path.join(os.path.abspath('.'), 'src')
if _src not in sys.path:
    sys.path.insert(0, _src)

from fm_copilot.pipeline import run_report

os.makedirs('output', exist_ok=True)
market_file = 'data_uploads/market.html'

config = {
    'squad_file':            'data_uploads/squad.html',
    'market_file':           market_file if os.path.exists(market_file) else '',
    'roles_file':            'data/roles.yaml',
    'archetypes_file':       'data/archetypes.yaml',
    'attribute_keys_file':   'data/attribute-keys.yaml',
    'context_dir':           'context',
    'dof_mode':              dof_mode,
    'api_key':               api_key if api_key else '',
    'model':                 'claude-sonnet-4-6',
    'output_file':           'output/report.html',
    'candidate_threshold':   60,
    'candidates_per_position': 3,
}

report_path = run_report(config)
print(f'\n✅ Report saved to {report_path}')


In [ ]:
#@title Step 6: View and download your report { display-mode: "form" }
from IPython.display import HTML, display
from google.colab import files

print('📄 Displaying report below...')
print('   (It may look best at 100% zoom in your browser)')
print()

with open(report_path, encoding='utf-8') as f:
    report_html = f.read()

# Show inline
display(HTML(f'<div style="height:700px;overflow:auto;border:1px solid #333;border-radius:8px">{report_html}</div>'))

# Offer download
print('\nDownloading report...')
files.download(report_path)

---

## 🔄 Generating another report

To generate a new report after a transfer window:

1. Re-upload your updated `squad.html` in **Step 2**
2. Update your club config / tactical notes in **Step 3** if anything changed
3. Re-run **Step 5** and **Step 6**

Your context is saved for the session — you only need to re-enter what's changed.

---

Made with ⚽ by [Michael Laweh](https://github.com/laweh-dev/fm24-sporting-director) · v0.2.0  
*Not affiliated with Sports Interactive or SEGA.*